# Bear Data Cleaning Pipeline

This notebook loads and cleans bear individual data from the Excel file 'Dades os_1996_2024.xlsx'.
It extracts the first sheet containing individual bear information and exports it to CSV format.

## 1. Configuration and Imports

In [52]:
!pip install openpyxl

In [53]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

import warnings
warnings.filterwarnings('ignore')

# Configuration: Paths and file names
NOTEBOOK_DIR = Path.cwd()
INPUT_FILE = NOTEBOOK_DIR / "Dades os_1996_2024.xlsx"
OUTPUT_DIR = NOTEBOOK_DIR / "cleaned_data"

# Create output directory if it doesn't exist
OUTPUT_DIR.mkdir(exist_ok=True)

# Excel sheet names
INDIVIDUALS_SHEET = "Individus_Totals_2024"  # Adjust if needed

print(f"Input file: {INPUT_FILE}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Input file exists: {INPUT_FILE.exists()}")

Input file: /home/aniol-garriga-torra-boss/Escriptori/ANIOL/UNI/4t Carrera/TFG/TFG-pirineus_raster/notebooks/Dades os_1996_2024.xlsx
Output directory: /home/aniol-garriga-torra-boss/Escriptori/ANIOL/UNI/4t Carrera/TFG/TFG-pirineus_raster/notebooks/cleaned_data
Input file exists: True


## 2. Load and Inspect Data

In [54]:
# Load all sheet names to identify the individuals sheet
xls = pd.ExcelFile(INPUT_FILE)
print("Available sheet names:")
print(xls.sheet_names)

# Load the first sheet (individuals data)
df_raw = pd.read_excel(INPUT_FILE, sheet_name=0)

print(f"\nShape of raw data: {df_raw.shape}")
print(f"\nColumn names:")
print(df_raw.columns.tolist())
print(f"\nFirst few rows:")
df_raw.head()

Available sheet names:
['Individus_Totals_2024', 'Dades GPS', 'Total_os_bru_1996_2024']

Shape of raw data: (194, 28)

Column names:
['Code commun / Código común', 'Nom Ours / Nombre Oso', 'Génotype Antagene / Genotipo Antagene', 'Genotype UAB', 'Sexe / Sexo', 'Année Naissance / Año de nacimiento', 'Mère / Madre', 'Codi mare', 'Père / Padre', 'Codi pare', 'Année de Mortalité / Ano de Mortalidad', 'Année de disparition supposée / Año de la presunta desaparición', 'Age / Edad', 'Detectado en 2022', 'Detectado en 2023', 'Detectado en 2024', datetime.datetime(2024, 2, 1, 0, 0), datetime.datetime(2024, 3, 1, 0, 0), datetime.datetime(2024, 4, 1, 0, 0), datetime.datetime(2024, 5, 1, 0, 0), datetime.datetime(2024, 6, 1, 0, 0), datetime.datetime(2024, 7, 1, 0, 0), datetime.datetime(2024, 8, 1, 0, 0), datetime.datetime(2024, 9, 1, 0, 0), datetime.datetime(2024, 10, 1, 0, 0), datetime.datetime(2024, 11, 1, 0, 0), datetime.datetime(2024, 12, 1, 0, 0), 'Total 2024']

First few rows:


,Code commun / Código común,Nom Ours / Nombre Oso,Génotype Antagene / Genotipo Antagene,Genotype UAB,Sexe / Sexo,Année Naissance / Año de nacimiento,Mère / Madre,Codi mare,Père / Padre,Codi pare,...,2024-04-01 00:00:00,2024-05-01 00:00:00,2024-06-01 00:00:00,2024-07-01 00:00:00,2024-08-01 00:00:00,2024-09-01 00:00:00,2024-10-01 00:00:00,2024-11-01 00:00:00,2024-12-01 00:00:00,Total 2024
0,NaN,Papillon,NaN,NaN,M,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
1,NaN,Cannelle,S2-PYR6,NaN,F,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2,NaN,Camille / Aspe-Ouest,S1-PYR4,Camille,M,1998,Cannelle,Sense codi,Papillon,Sense codi,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3,NaN,Ourson mort,NaN,NaN,M,2000,Cannelle,Sense codi,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
4,F001,Ziva,S8-SLO13,NaN,F,1990,Slovène,Sense codi,Slovène,Sense codi,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [55]:
# Diagnostic: Check column types
print("Column types and values:")
for i, col in enumerate(df_raw.columns):
    print(f"{i:2d}. Type: {type(col).__name__:15s} | Value: {str(col)[:60]}")

Column types and values:
 0. Type: str             | Value: Code commun / Código común
 1. Type: str             | Value: Nom Ours / Nombre Oso
 2. Type: str             | Value: Génotype Antagene / Genotipo Antagene
 3. Type: str             | Value: Genotype UAB
 4. Type: str             | Value: Sexe / Sexo
 5. Type: str             | Value: Année Naissance / Año de nacimiento
 6. Type: str             | Value: Mère / Madre
 7. Type: str             | Value: Codi mare
 8. Type: str             | Value: Père / Padre
 9. Type: str             | Value: Codi pare
10. Type: str             | Value: Année de Mortalité / Ano de Mortalidad
11. Type: str             | Value: Année de disparition supposée / Año de la presunta desaparic
12. Type: str             | Value: Age / Edad
13. Type: str             | Value: Detectado en 2022
14. Type: str             | Value: Detectado en 2023
15. Type: str             | Value: Detectado en 2024
16. Type: datetime        | Value: 2024-02-01 00:00:00
1

In [56]:
# Check data types and missing values
print("Data types:")
print(df_raw.dtypes)
print(f"\nMissing values:")
print(df_raw.isnull().sum())
print(f"\nBasic statistics:")
df_raw.describe()

Data types:
Code commun / Código común                                             str
Nom Ours / Nombre Oso                                                  str
Génotype Antagene / Genotipo Antagene                                  str
Genotype UAB                                                           str
Sexe / Sexo                                                            str
Année Naissance / Año de nacimiento                                 object
Mère / Madre                                                           str
Codi mare                                                              str
Père / Padre                                                           str
Codi pare                                                              str
Année de Mortalité / Ano de Mortalidad                             float64
Année de disparition supposée / Año de la presunta desaparición    float64
Age / Edad                                                         float64
Detectado en 

,Année de Mortalité / Ano de Mortalidad,Année de disparition supposée / Año de la presunta desaparición,Age / Edad,Detectado en 2022,Detectado en 2023,Detectado en 2024,2024-02-01 00:00:00,2024-03-01 00:00:00,2024-04-01 00:00:00,2024-05-01 00:00:00,2024-06-01 00:00:00,2024-07-01 00:00:00,2024-08-01 00:00:00,2024-09-01 00:00:00,2024-10-01 00:00:00,2024-11-01 00:00:00,2024-12-01 00:00:00,Total 2024
count,35.000000,36.000000,121.000000,143.000000,160.000000,183.000000,10.000000,15.000000,24.000000,38.000000,42.000000,58.000000,55.000000,41.000000,38.000000,11.000000,3.000000,182.000000
mean,2013.885714,2016.777778,5.057851,0.531469,0.518750,1.049180,1.100000,1.600000,2.250000,3.131579,2.738095,2.448276,2.254545,2.097561,3.131579,1.727273,1.666667,4.494505
std,7.745099,5.986228,5.359256,0.500763,0.501217,7.075168,0.316228,1.352247,1.481773,3.094643,2.479767,2.027637,1.797304,1.578051,3.362520,1.009050,0.577350,7.187887
min,1997.000000,2002.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000
25%,2008.000000,2013.750000,1.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.500000,0.000000
50%,2017.000000,2018.500000,3.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,1.500000,1.000000,2.000000,1.000000
75%,2020.000000,2022.000000,8.000000,1.000000,1.000000,1.000000,1.000000,1.500000,3.000000,4.000000,3.750000,3.000000,3.000000,3.000000,4.500000,3.000000,2.000000,6.000000
max,2022.000000,2023.000000,27.000000,1.000000,1.000000,96.000000,2.000000,6.000000,6.000000,16.000000,11.000000,10.000000,11.000000,8.000000,16.000000,3.000000,2.000000,42.000000


## 3. Data Cleaning and Standardization

In [57]:
column_mapping = {
    'Code commun / Código común': 'code',
    'Nom Ours / Nombre Oso': 'name',
    'Génotype Antagene / Genotipo Antagene': 'genotype',
    'Genotype UAB': 'genotype_uab',
    'Sexe / Sexo': 'sex',
    'Année Naissance / Año de nacimiento': 'born_year',
    'Mère / Madre': 'mum_name',
    'Codi mare': 'mum_code',
    'Père / Padre': 'father_name',
    'Codi pare': 'father_code',
    'Année de Mortalité / Ano de Mortalidad': 'mortality_year',
    'Année de disparition supposée / Año de la presunta desaparición': 'suposed_desaparition_year',
    'Age / Edad': 'age',
    'Detectado en 2022': 'detected_2022',
    'Detectado en 2023': 'detected_2023',
    'Detectado en 2024': 'detected_2024',
}

# Map datetime columns to month names
datetime_to_month = {
    2: 'num_detections_feb_24',
    3: 'num_detections_mar_24',
    4: 'num_detections_apr_24',
    5: 'num_detections_may_24',
    6: 'num_detections_jun_24',
    7: 'num_detections_jul_24',
    8: 'num_detections_aug_24',
    9: 'num_detections_sep_24',
    10: 'num_detections_oct_24',
    11: 'num_detections_nov_24',
    12: 'num_detections_dec_24',
}

# Find and map datetime columns (they're datetime.datetime objects)
for col in df_raw.columns:
    if isinstance(col, datetime):
        month = col.month
        if month in datetime_to_month:
            column_mapping[col] = datetime_to_month[month]

# Find and map the total column
for col in df_raw.columns:
    if isinstance(col, str) and 'Total' in col:
        column_mapping[col] = 'num_detections_total_24'

print("Column mapping created:")
for original, mapped in column_mapping.items():
    if isinstance(original, datetime):
        col_str = f"Timestamp({original.month:02d}/2024)"
    else:
        col_str = str(original)[:50]
    print(f"  {col_str:50s} → {mapped}")

# Get only columns that exist in the dataframe, preserving their order
columns_to_select = [col for col in df_raw.columns if col in column_mapping.keys()]
print(f"\nColumns found: {len(columns_to_select)}/{len(column_mapping)}")

Column mapping created:
  Code commun / Código común                         → code
  Nom Ours / Nombre Oso                              → name
  Génotype Antagene / Genotipo Antagene              → genotype
  Genotype UAB                                       → genotype_uab
  Sexe / Sexo                                        → sex
  Année Naissance / Año de nacimiento                → born_year
  Mère / Madre                                       → mum_name
  Codi mare                                          → mum_code
  Père / Padre                                       → father_name
  Codi pare                                          → father_code
  Année de Mortalité / Ano de Mortalidad             → mortality_year
  Année de disparition supposée / Año de la presunta → suposed_desaparition_year
  Age / Edad                                         → age
  Detectado en 2022                                  → detected_2022
  Detectado en 2023                                  → dete

In [58]:
df_clean = df_raw[columns_to_select].copy()

# Rename columns according to mapping
df_clean = df_clean.rename(columns=column_mapping)

# Filter to keep only valid individual records (rows 2-183 in Excel = indices 1-182 in Python)
print(f"Data shape before filtering: {df_clean.shape}")
df_clean = df_clean.iloc[:182]  # Keep only rows 2-183 from Excel (1-based indexing)
print(f"Data shape after filtering to valid individuals: {df_clean.shape}")
print(f"  Kept rows: 1-183 (Excel indexing) = indices 0-182 (Python indexing)")

# Reorder columns to the exact order specified
desired_column_order = [
    'code', 'name', 'genotype', 'genotype_uab', 'sex', 'born_year',
    'mum_name', 'mum_code', 'father_name', 'father_code',
    'mortality_year', 'suposed_desaparition_year', 'age',
    'detected_2022', 'detected_2023', 'detected_2024',
    'num_detections_feb_24', 'num_detections_mar_24', 'num_detections_apr_24',
    'num_detections_may_24', 'num_detections_jun_24', 'num_detections_jul_24',
    'num_detections_aug_24', 'num_detections_sep_24', 'num_detections_oct_24',
    'num_detections_nov_24', 'num_detections_dec_24', 'num_detections_total_24'
]

# Select and reorder
available_columns = [col for col in desired_column_order if col in df_clean.columns]
df_clean = df_clean[available_columns]

print(f"\nDataframe shape: {df_clean.shape}")
print(f"\nFinal column order ({len(df_clean.columns)} columns):")
for i, col in enumerate(df_clean.columns, 1):
    print(f"  {i:2d}. {col}")

Data shape before filtering: (194, 28)
Data shape after filtering to valid individuals: (182, 28)
  Kept rows: 1-183 (Excel indexing) = indices 0-182 (Python indexing)

Dataframe shape: (182, 28)

Final column order (28 columns):
   1. code
   2. name
   3. genotype
   4. genotype_uab
   5. sex
   6. born_year
   7. mum_name
   8. mum_code
   9. father_name
  10. father_code
  11. mortality_year
  12. suposed_desaparition_year
  13. age
  14. detected_2022
  15. detected_2023
  16. detected_2024
  17. num_detections_feb_24
  18. num_detections_mar_24
  19. num_detections_apr_24
  20. num_detections_may_24
  21. num_detections_jun_24
  22. num_detections_jul_24
  23. num_detections_aug_24
  24. num_detections_sep_24
  25. num_detections_oct_24
  26. num_detections_nov_24
  27. num_detections_dec_24
  28. num_detections_total_24


In [59]:
# Fill missing codes for the first 4 rows with correct bear names
# Using .loc[] for reliable pandas assignment
df_clean.loc[0, 'code'] = 'Papillon'
df_clean.loc[1, 'code'] = 'Cannelle'
df_clean.loc[2, 'code'] = 'Camille'
df_clean.loc[3, 'code'] = 'Ourson'

print("✓ Codes assigned to first 4 rows:")
print(df_clean.loc[1:4, 'code'].to_string())

✓ Codes assigned to first 4 rows:
1    Cannelle
2     Camille
3      Ourson
4        F001


In [60]:
df_clean.head()

,code,name,genotype,genotype_uab,sex,born_year,mum_name,mum_code,father_name,father_code,...,num_detections_apr_24,num_detections_may_24,num_detections_jun_24,num_detections_jul_24,num_detections_aug_24,num_detections_sep_24,num_detections_oct_24,num_detections_nov_24,num_detections_dec_24,num_detections_total_24
0,Papillon,Papillon,NaN,NaN,M,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
1,Cannelle,Cannelle,S2-PYR6,NaN,F,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2,Camille,Camille / Aspe-Ouest,S1-PYR4,Camille,M,1998,Cannelle,Sense codi,Papillon,Sense codi,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3,Ourson,Ourson mort,NaN,NaN,M,2000,Cannelle,Sense codi,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
4,F001,Ziva,S8-SLO13,NaN,F,1990,Slovène,Sense codi,Slovène,Sense codi,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [61]:
# Data type conversions and cleaning

# String columns - handle null/NaN values
string_columns = ['code', 'name', 'genotype', 'genotype_uab', 'mum_name', 'mum_code', 'father_name', 'father_code']
for col in string_columns:
    if col in df_clean.columns:
        # Convert to string and handle NaN
        df_clean[col] = df_clean[col].astype(str).replace('nan', pd.NA)
        # Strip whitespace
        df_clean[col] = df_clean[col].apply(lambda x: x.strip() if pd.notna(x) else pd.NA)

# Sex should be categorical (M, F, I) - uppercase
if 'sex' in df_clean.columns:
    df_clean['sex'] = df_clean['sex'].astype(str).str.upper().replace('nan', pd.NA).str.strip()
    # Validate sex values
    valid_sex = df_clean['sex'].isin(['M', 'F', 'I']) | df_clean['sex'].isna()
    if not valid_sex.all():
        invalid = df_clean[~valid_sex]['sex'].unique()
        print(f"Warning: Invalid sex values found: {invalid}")

# Year columns (as string to preserve format)
year_columns = ['born_year', 'mortality_year', 'suposed_desaparition_year']
for col in year_columns:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str).replace('nan', pd.NA).str.strip()

# Age column (integer or null)
if 'age' in df_clean.columns:
    df_clean['age'] = pd.to_numeric(df_clean['age'], errors='coerce').astype('Int64')

# Binary detection columns (0 or 1)
detection_columns = ['detected_2022', 'detected_2023', 'detected_2024']
for col in detection_columns:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').astype('Int64')
        # Validate binary values
        valid_binary = df_clean[col].isin([0, 1]) | df_clean[col].isna()
        if not valid_binary.all():
            print(f"Warning: {col} contains non-binary values")

# Monthly detection count columns (integer or null)
monthly_cols = [col for col in df_clean.columns if col.startswith('num_detections_') and col != 'num_detections_total_24']
for col in monthly_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').astype('Int64')

# Total detections column (integer)
if 'num_detections_total_24' in df_clean.columns:
    df_clean['num_detections_total_24'] = pd.to_numeric(df_clean['num_detections_total_24'], errors='coerce').astype('Int64')

print("✓ Data type conversions completed")
print(f"\nData types:")
print(df_clean.dtypes)

['F?']
Length: 1, dtype: str
✓ Data type conversions completed

Data types:
code                           str
name                           str
genotype                       str
genotype_uab                   str
sex                            str
born_year                      str
mum_name                       str
mum_code                       str
father_name                    str
father_code                    str
mortality_year                 str
suposed_desaparition_year      str
age                          Int64
detected_2022                Int64
detected_2023                Int64
detected_2024                Int64
num_detections_feb_24        Int64
num_detections_mar_24        Int64
num_detections_apr_24        Int64
num_detections_may_24        Int64
num_detections_jun_24        Int64
num_detections_jul_24        Int64
num_detections_aug_24        Int64
num_detections_sep_24        Int64
num_detections_oct_24        Int64
num_detections_nov_24        Int64
num_detections

In [62]:
# Data Validation and Quality Checks
print("=== DATA VALIDATION REPORT ===\n")

# Check for required code column
if 'code' in df_clean.columns:
    missing_codes = df_clean['code'].isna().sum()
    print(f"Missing codes: {missing_codes}")
    if missing_codes > 0:
        print(f"  Rows with missing codes: {df_clean[df_clean['code'].isna()].index.tolist()}")

# Check for duplicates
duplicate_codes = df_clean['code'].duplicated().sum()
print(f"Duplicate codes: {duplicate_codes}")
if duplicate_codes > 0:
    print(f"  Duplicate codes: {df_clean[df_clean['code'].duplicated(keep=False)]['code'].unique().tolist()}")

# Check sex distribution
if 'sex' in df_clean.columns:
    print(f"\nSex distribution:")
    print(df_clean['sex'].value_counts(dropna=False))

# Check detection columns
detection_cols = ['detected_2022', 'detected_2023', 'detected_2024']
detection_cols_present = [col for col in detection_cols if col in df_clean.columns]
if detection_cols_present:
    print(f"\nDetection summary:")
    for col in detection_cols_present:
        present = (df_clean[col] == 1).sum()
        print(f"  {col}: {present} individuals detected")

# Check monthly detection counts sum to total
if 'num_detections_total_24' in df_clean.columns:
    monthly_cols = [col for col in df_clean.columns if col.startswith('num_detections_') and col != 'num_detections_total_24']
    monthly_sum = df_clean[monthly_cols].sum(axis=1)
    total_col = df_clean['num_detections_total_24']
    
    # Compare (handle NaN cases)
    comparison = (monthly_sum == total_col) | (monthly_sum.isna() & total_col.isna())
    mismatches = (~comparison).sum()
    print(f"\nMonthly detections sum validation:")
    print(f"  Rows where monthly sum matches total: {comparison.sum()}")
    print(f"  Rows with mismatches: {mismatches}")
    if mismatches > 0:
        print(f"  Using calculated sum as total (Excel formulas may have been source)")

# Missing value summary
print(f"\nMissing values per column:")
missing_summary = df_clean.isnull().sum()
missing_summary = missing_summary[missing_summary > 0]
if len(missing_summary) > 0:
    print(missing_summary)
else:
    print("  No missing values")

print(f"\nTotal rows: {len(df_clean)}")
print(f"Total columns: {len(df_clean.columns)}")

=== DATA VALIDATION REPORT ===

Missing codes: 0
Duplicate codes: 0

Sex distribution:
sex
F     84
M     73
I     24
F?     1
Name: count, dtype: int64

Detection summary:
  detected_2022: 76 individuals detected
  detected_2023: 83 individuals detected
  detected_2024: 96 individuals detected

Monthly detections sum validation:
  Rows where monthly sum matches total: 182
  Rows with mismatches: 0

Missing values per column:
name                         115
genotype                      31
genotype_uab                 145
born_year                      2
mum_name                       2
mum_code                       2
father_name                   25
father_code                   25
mortality_year               147
suposed_desaparition_year    146
age                           61
detected_2022                 39
detected_2023                 22
num_detections_feb_24        172
num_detections_mar_24        167
num_detections_apr_24        158
num_detections_may_24        144
num_detec

## 4. Export Cleaned Data

In [63]:
# Export individual bear data to CSV
output_file = OUTPUT_DIR / "bear_individuals.csv"

df_clean.to_csv(output_file, index=False)

print(f"✓ Cleaned data exported to: {output_file}")
print(f"  - Rows: {len(df_clean)}")
print(f"  - Columns: {len(df_clean.columns)}")
print(f"\nColumn list:")
for i, col in enumerate(df_clean.columns, 1):
    print(f"  {i:2d}. {col}")

✓ Cleaned data exported to: /home/aniol-garriga-torra-boss/Escriptori/ANIOL/UNI/4t Carrera/TFG/TFG-pirineus_raster/notebooks/cleaned_data/bear_individuals.csv
  - Rows: 182
  - Columns: 28

Column list:
   1. code
   2. name
   3. genotype
   4. genotype_uab
   5. sex
   6. born_year
   7. mum_name
   8. mum_code
   9. father_name
  10. father_code
  11. mortality_year
  12. suposed_desaparition_year
  13. age
  14. detected_2022
  15. detected_2023
  16. detected_2024
  17. num_detections_feb_24
  18. num_detections_mar_24
  19. num_detections_apr_24
  20. num_detections_may_24
  21. num_detections_jun_24
  22. num_detections_jul_24
  23. num_detections_aug_24
  24. num_detections_sep_24
  25. num_detections_oct_24
  26. num_detections_nov_24
  27. num_detections_dec_24
  28. num_detections_total_24


## 5. Preview of Cleaned Data

In [64]:
# Display first rows of cleaned data
print("First 10 rows of cleaned bear individual data:\n")
df_clean.head(10)

First 10 rows of cleaned bear individual data:



,code,name,genotype,genotype_uab,sex,born_year,mum_name,mum_code,father_name,father_code,...,num_detections_apr_24,num_detections_may_24,num_detections_jun_24,num_detections_jul_24,num_detections_aug_24,num_detections_sep_24,num_detections_oct_24,num_detections_nov_24,num_detections_dec_24,num_detections_total_24
0,Papillon,Papillon,NaN,NaN,M,NaN,NaN,NaN,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0
1,Cannelle,Cannelle,S2-PYR6,NaN,F,NaN,NaN,NaN,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0
2,Camille,Camille / Aspe-Ouest,S1-PYR4,Camille,M,1998,Cannelle,Sense codi,Papillon,Sense codi,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0
3,Ourson,Ourson mort,NaN,NaN,M,2000,Cannelle,Sense codi,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0
4,F001,Ziva,S8-SLO13,NaN,F,1990,Slovène,Sense codi,Slovène,Sense codi,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0
5,F002,Mellba,NaN,NaN,F,1991,Slovène,Sense codi,Slovène,Sense codi,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0
6,M003,Pyros,S1-SLO1,NaN,M,1988,Slovène,Sense codi,Slovène,Sense codi,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0
7,M004,Nere,S2-SL06,NaN,M,1997,Ziva,F001,Slovène,Sense codi,...,2,3,5,4,1,1,1,<NA>,<NA>,23
8,M005,Kouki,NaN,NaN,M,1997,Ziva,F001,Pyros,M003,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0
9,M006,Boutxy,S1-SLO2,NaN,M,1997,Mellba,F002,Pyros,M003,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0
